# Module 1 – Données : chargement et inspection

Charger les avis (`recommendations.csv`) et les deux fichiers de jeux (Kozyriev `games.csv` et FronkonGames `games_1.csv`), puis regarder : taille, types, valeurs manquantes, doublons.

On ne supprime et on n'impute **rien** à cette étape.

In [12]:
import sys
sys.path.append("..")

import pandas as pd
from config import *

NROWS = 5_000_000  # échantillon de développement (le fichier complet fait 41 M de lignes)

## 1. Avis

In [13]:
reviews = pd.read_csv(
    RAW_DIR / "recommendations.csv",
    usecols=["app_id", "date", "is_recommended", "hours", "helpful"],
    dtype={"app_id": "int32", "is_recommended": "bool", "hours": "float32", "helpful": "int32"},
    nrows=NROWS,
)
reviews["date"] = pd.to_datetime(reviews["date"], format="%Y-%m-%d", errors="coerce")

print("Shape :", reviews.shape)
reviews.dtypes

Shape : (5000000, 5)


app_id                     int32
helpful                    int32
date              datetime64[us]
is_recommended              bool
hours                    float32
dtype: object

In [14]:
print("% manquant par colonne :")
print(reviews.isna().mean() * 100)
print("Lignes en double :", reviews.duplicated().sum())

% manquant par colonne :
app_id            0.0
helpful           0.0
date              0.0
is_recommended    0.0
hours             0.0
dtype: float64
Lignes en double : 185657


Les « lignes en double » ne sont pas forcément de vrais doublons : deux joueurs peuvent laisser un avis sur le même jeu, le même jour, avec les mêmes heures de jeu. On ne les supprime donc pas.

## 2. Jeux Kozyriev (`games.csv`)

In [15]:
games = pd.read_csv(RAW_DIR / "games.csv")
games["date_release"] = pd.to_datetime(games["date_release"], format="%Y-%m-%d", errors="coerce")

print("Shape :", games.shape)
games.dtypes

Shape : (50872, 13)


app_id                     int64
title                        str
date_release      datetime64[us]
win                         bool
mac                         bool
linux                       bool
rating                       str
positive_ratio             int64
user_reviews               int64
price_final              float64
price_original           float64
discount                 float64
steam_deck                  bool
dtype: object

In [16]:
print("% manquant par colonne :")
print(games.isna().mean() * 100)
print("app_id en double :", games["app_id"].duplicated().sum())

% manquant par colonne :
app_id            0.0
title             0.0
date_release      0.0
win               0.0
mac               0.0
linux             0.0
rating            0.0
positive_ratio    0.0
user_reviews      0.0
price_final       0.0
price_original    0.0
discount          0.0
steam_deck        0.0
dtype: float64
app_id en double : 0


## 3. Jeux FronkonGames (`games_1.csv`)

L'en-tête du fichier est décalé : `Discount` et `DLC count` y sont collées en une seule colonne, `DiscountDLC count`, alors que chaque ligne contient bien les deux valeurs. On corrige les noms de colonnes avant de lire le fichier, sinon toutes les colonnes après `Price` sont décalées d'un cran.

On ne garde que les colonnes utiles (`usecols`), car le fichier est lourd (400 Mo).

In [17]:
path = RAW_DIR / "games_1.csv"

# Correction de l'en-tête : "DiscountDLC count" -> "Discount", "DLC count"
cols = pd.read_csv(path, nrows=0).columns.tolist()
i = cols.index("DiscountDLC count")
cols[i:i + 1] = ["Discount", "DLC count"]

fronkon = pd.read_csv(
    path,
    names=cols,
    skiprows=1,  # on saute l'en-tête d'origine, remplacé par cols
    usecols=["AppID", "Name", "Release date", "Required age", "Price", "DLC count",
             "About the game", "Supported languages", "Windows", "Mac", "Linux",
             "Categories", "Genres", "Tags"],
)
fronkon["Release date"] = pd.to_datetime(fronkon["Release date"], format="%b %d, %Y", errors="coerce")

print("Shape :", fronkon.shape)
fronkon.head(3)  # vérifier que les valeurs sont dans les bonnes colonnes

Shape : (125855, 14)


,AppID,Name,Release date,Required age,Price,DLC count,About the game,Supported languages,Windows,Mac,Linux,Categories,Genres,Tags
0,2539430,Black Dragon Mage Playtest,2023-08-01,0,0.00,0,NaN,[],True,False,False,NaN,NaN,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,2016-07-29,0,5.24,0,"Springtime, April: when the cherry trees come ...",['English'],True,False,False,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute"
2,1034400,Mystery Solitaire The Black Raven,2019-05-06,0,4.99,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",True,True,False,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec..."


Les dates sont au format américain (`Aug 1, 2023`). Une date qui ne suit pas ce format (par exemple `Aug 2023` ou `Coming soon`) devient `NaT` : elle apparaît dans le % de valeurs manquantes de `Release date` ci-dessous, et n'est pas supprimée.

In [18]:
print("% manquant par colonne :")
print(fronkon.isna().sum())
print("AppID en double :", fronkon["AppID"].duplicated().sum())

% manquant par colonne :
AppID                      0
Name                       1
Release date               0
Required age               0
Price                      0
DLC count                  0
About the game          8462
Supported languages        0
Windows                    0
Mac                        0
Linux                      0
Categories              8966
Genres                  8423
Tags                   42502
dtype: int64
AppID en double : 0


## 4. `reviews_clean` : avis du sous-ensemble de jeux

On tire un sous-ensemble de jeux **par `app_id`** (jeux sortis entre 2015 et 2021), puis on garde **tous** leurs avis. On n'échantillonne jamais les lignes d'avis : cela fausserait le nombre d'avis par jeu, donc la cible.

La date de sortie vient de Kozyriev (`date_release`), qui est notre table de base.

In [19]:
N_GAMES = 10_000  # nombre de jeux tirés au sort (None = tous les jeux 2015-2021)

subset = games[(games["date_release"] >= "2015-01-01") & (games["date_release"] <= "2021-12-31")]
if N_GAMES is not None:
    subset = subset.sample(N_GAMES, random_state=SEED)

release = subset.set_index("app_id")["date_release"]  # app_id -> date de sortie
print("Jeux retenus :", len(release))

Jeux retenus : 10000


In [20]:
# Le fichier complet ne tient pas en mémoire : on le lit par morceaux
# et on ne garde que les avis des jeux retenus.
chunks = []
for chunk in pd.read_csv(
    RAW_DIR / "recommendations.csv",
    usecols=["app_id", "date", "is_recommended", "hours", "helpful"],
    dtype={"app_id": "int32", "is_recommended": "bool", "hours": "float32", "helpful": "int32"},
    chunksize=2_000_000,
):
    chunks.append(chunk[chunk["app_id"].isin(release.index)])

reviews_clean = pd.concat(chunks, ignore_index=True)
reviews_clean["date"] = pd.to_datetime(reviews_clean["date"], format="%Y-%m-%d", errors="coerce")
assert reviews_clean["date"].notna().all(), "des dates d'avis sont illisibles"

print("Avis des jeux retenus :", len(reviews_clean))
print("Jeux avec au moins 1 avis :", reviews_clean["app_id"].nunique())

Avis des jeux retenus : 7304209
Jeux avec au moins 1 avis : 7906


### `days` : nombre de jours entre la sortie du jeu et l'avis

Un avis avec `days < 0` a été posté **avant** la date de sortie officielle, en général pendant l'accès anticipé. On les compte, puis on les exclut : la cible et les signaux précoces se mesurent à partir de la sortie.

In [21]:
reviews_clean["days"] = (reviews_clean["date"] - reviews_clean["app_id"].map(release)).dt.days

n_before = (reviews_clean["days"] < 0).sum()
print(f"Avis avant la sortie (days < 0) : {n_before} ({n_before / len(reviews_clean):.2%}) -> exclus")
reviews_clean = reviews_clean[reviews_clean["days"] >= 0]

print("Avis gardés :", len(reviews_clean))
reviews_clean.head()

Avis avant la sortie (days < 0) : 470961 (6.45%) -> exclus
Avis gardés : 6833248


,app_id,helpful,date,is_recommended,hours,days
0,306130,0,2021-10-10,True,8.600000,1602
1,255710,0,2021-05-21,True,18.700001,2264
2,306130,0,2021-05-12,True,61.099998,1451
3,635260,0,2022-01-30,True,177.000000,1535
4,518790,0,2022-12-31,True,10.000000,2144


In [22]:
reviews_clean = reviews_clean[["app_id", "date", "days", "is_recommended", "hours", "helpful"]]
reviews_clean.to_parquet(PROC_DIR / "reviews_clean.parquet", index=False)
print("Enregistré :", PROC_DIR / "reviews_clean.parquet")

Enregistré : C:\Users\famil\Documents\ESILV\A4\S1\Machine Learning\SteamGameIA\data\processed\reviews_clean.parquet


## 5. `games_clean` : fusion Kozyriev + FronkonGames

On fusionne le sous-ensemble de jeux (table de base Kozyriev) avec FronkonGames sur `app_id` :
- **jointure `left`** : on garde tous les jeux tirés, même ceux absents de FronkonGames (leurs colonnes FronkonGames restent vides) ;
- **`validate="one_to_one"`** : pandas lève une erreur si un `app_id` est en double d'un côté, ce qui dupliquerait des lignes ;
- **`indicator=True`** : la colonne `_merge` indique, pour chaque jeu, s'il a été trouvé dans FronkonGames.

`games_clean` contient **tous** les jeux tirés, y compris ceux sans aucun avis. Le module 2 en a besoin pour leur donner 0 avis.

In [23]:
fronkon = fronkon.rename(columns={"AppID": "app_id"})
merged = subset.merge(fronkon, on="app_id", how="left", validate="one_to_one", indicator=True)
assert merged["app_id"].is_unique

print("Jeux du sous-ensemble :", len(subset))
print("Jeux après la fusion  :", len(merged))
print(merged["_merge"].value_counts())  # both = trouvé dans FronkonGames, left_only = absent

# Comparaison des deux dates de sortie
diff = (merged["Release date"] - merged["date_release"]).dt.days.abs()
print("Dates de sortie différentes de plus de 30 jours :", (diff > 30).sum())

Jeux du sous-ensemble : 10000
Jeux après la fusion  : 10000
_merge
both          7797
left_only     2203
right_only       0
Name: count, dtype: int64
Dates de sortie différentes de plus de 30 jours : 143


### Colonnes gardées

- **Gardées :** identifiant, titre, date de sortie (Kozyriev), prix de base `price_original`, plateformes, et les colonnes FronkonGames (âge requis, nombre de DLC, description, langues, catégories, genres, tags).
- **Exclues, car mesurées après la sortie (fuite de données) :** `rating`, `positive_ratio`, `user_reviews`, `price_final`, `discount`, `steam_deck`, ainsi que `Price` de FronkonGames (prix au moment de la collecte, pas au lancement).
- **Exclues, car en double :** `Name` (= `title`), `Windows` / `Mac` / `Linux` (= `win` / `mac` / `linux`), `Release date` (on garde la date Kozyriev).

⚠️ `dlc_count` et `tags` ont pu changer après la sortie : c'est un risque de fuite, à discuter dans le module 3.

In [24]:
games_clean = merged.rename(columns={
    "date_release": "release_date",
    "Required age": "required_age",
    "DLC count": "dlc_count",
    "About the game": "description",
    "Supported languages": "languages",
    "Categories": "categories",
    "Genres": "genres",
    "Tags": "tags",
})[["app_id", "title", "release_date", "price_original", "win", "mac", "linux",
    "required_age", "dlc_count", "description", "languages", "categories", "genres", "tags"]]

games_clean.to_parquet(PROC_DIR / "games_clean.parquet", index=False)
print("Enregistré :", PROC_DIR / "games_clean.parquet", games_clean.shape)

Enregistré : C:\Users\famil\Documents\ESILV\A4\S1\Machine Learning\SteamGameIA\data\processed\games_clean.parquet (10000, 14)
